<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_170_Variables_aleatorias_discretas_binomial_y_Poisson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 170. Variables aleatorias discretas: distribuciones binomial y de Poisson</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 4. Probabilidad**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Retos en la app:** [las tres misiones del Tema 170](https://fborrasumh.github.io/bioestadistica/#tema-170)

**Evaluación:** el examen del Módulo 4 se hace **a mano**. Resuelve primero cada ejercicio con papel y calculadora, y usa el código solo para comprobar.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** usa **Entorno de ejecución → Ejecutar todo** antes de empezar, para que se dibujen los gráficos. Las celdas con borde azul son las diapositivas; el desarrollo de cada bloque está debajo y se puede plegar con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Definir variable aleatoria discreta, función de probabilidad y función de distribución.
2. Calcular la esperanza y la varianza de una variable discreta.
3. Reconocer cuándo un recuento clínico sigue una distribución **binomial** y calcular sus probabilidades.
4. Reconocer cuándo sigue una distribución de **Poisson** y calcular sus probabilidades.
5. Usar la distribución de Poisson para valorar si un número de casos es inusualmente alto.
6. Calcular estas probabilidades a mano y comprobarlas con `scipy.stats`.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: binomial y Poisson | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Ocho infecciones en marzo: ¿brote o azar?</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Si nada hubiera cambiado, 8 o más casos ocurrirían en 1 de cada 80 meses, aproximadamente.</li><li>Es raro bajo el modelo de azar: justifica investigar, no prueba un brote.</li></ul></div>

In [ ]:
#@title Figura: Ocho infecciones en marzo: ¿brote o azar?
from scipy import stats
import matplotlib.pyplot as plt

x = range(0, 13)
p = stats.poisson.pmf(x, 3)
plt.figure(figsize=(10, 4.5))
plt.bar(x, p, color=["firebrick" if k >= 8 else "steelblue" for k in x])
plt.xlabel("Infecciones de herida quirúrgica en un mes")
plt.ylabel("Probabilidad")
plt.title(f"Si la media sigue siendo 3: P(8 o más) = {stats.poisson.sf(7, 3):.3f}", fontweight="bold")
plt.xticks(x)
plt.show()

#### Desarrollo para el estudio

El servicio de Medicina Preventiva de un hospital registra una media de **3 infecciones de herida quirúrgica al mes** en Cirugía General. En marzo se registran **8**.

La supervisora pregunta si hay que activar el protocolo de **brote**. Ocho casos son más que tres, pero el número de infecciones varía de un mes a otro por puro azar: algunos meses habrá 1 y otros 5. La pregunta correcta es: **si nada hubiera cambiado, ¿con qué frecuencia veríamos 8 o más casos en un mes?**

Para responderla necesitamos un **modelo de probabilidad para recuentos**. Los recuentos (casos, ingresos, respuestas a un tratamiento, efectos adversos) son variables aleatorias discretas, y dos distribuciones describen la mayoría de ellos en medicina: la **binomial** y la de **Poisson**.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Variable aleatoria discreta</font>

Una **variable aleatoria** asigna un número a cada resultado de un experimento aleatorio. Es **discreta** si toma valores aislados, normalmente recuentos: 0, 1, 2…

| Concepto | Definición |
| --- | --- |
| Función de probabilidad | $P(X = k)$ para cada valor *k* |
| Función de distribución | $F(k) = P(X \le k)$, la probabilidad acumulada |
| Esperanza (media) | $E(X) = \sum k \cdot P(X = k)$ |
| Varianza | $\text{Var}(X) = \sum (k - E(X))^2 \cdot P(X = k)$ |

### <font color='steelblue'>2.2 La distribución binomial</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Binomial: aciertos en n ensayos independientes</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_170_binomial.png" width="780" alt="Binomial: aciertos en n ensayos independientes"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>n ensayos, misma probabilidad p, independientes: número de éxitos ~ B(n, p).</li><li>Media n·p; varianza n·p·(1 − p).</li></ul></div>

#### Desarrollo para el estudio

Cuenta el número de **éxitos** en *n* ensayos cuando:

1. El número de ensayos *n* es fijo.
2. Cada ensayo tiene dos resultados posibles (éxito o fracaso).
3. La probabilidad de éxito *p* es la misma en todos.
4. Los ensayos son independientes.

$$P(X = k) = \binom{n}{k}\, p^k\, (1-p)^{n-k} \qquad E(X) = np \qquad \text{Var}(X) = np(1-p)$$

**Ejemplos clínicos:** número de pacientes que responden a un tratamiento entre 20 tratados; número de hijos enfermos entre 3 de una pareja de portadores; número de preguntas acertadas por un modelo de IA entre 50.

### <font color='steelblue'>2.3 La distribución de Poisson</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Poisson: recuentos de sucesos raros</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_170_poisson.png" width="780" alt="Poisson: recuentos de sucesos raros"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Número de sucesos en un intervalo de tiempo o espacio, sin máximo fijo.</li><li>Media = varianza = λ. Si la varianza es mucho mayor, hay sobredispersión (contagios, conglomerados).</li></ul></div>

#### Desarrollo para el estudio

Cuenta el número de **sucesos** que ocurren en un intervalo de tiempo o de espacio cuando:

1. Los sucesos ocurren de forma independiente.
2. Ocurren con una tasa media constante λ.
3. Dos sucesos no ocurren exactamente a la vez.

$$P(X = k) = \frac{e^{-\lambda}\, \lambda^k}{k!} \qquad E(X) = \lambda \qquad \text{Var}(X) = \lambda$$

**Ejemplos clínicos:** infecciones al mes en un servicio, ingresos en la UCI por noche, casos anuales de una enfermedad rara en una provincia.

Cuando *n* es grande y *p* pequeña (por ejemplo, *n* ≥ 50 y *np* ≤ 5), la binomial se aproxima por una Poisson con λ = *np*.

## <font color='steelblue'>3. Demostración: binomial y Poisson (20 min)</font>

### <font color='steelblue'>3.1 Una variable discreta a mano: hijos enfermos</font>

Una pareja de portadores de fibrosis quística (Tema 140) tiene 3 hijos. *X* = número de hijos enfermos sigue una binomial con *n* = 3 y *p* = 1/4.

**A mano:**

| *k* | $\binom{3}{k}$ | $P(X = k)$ |
| --- | --- | --- |
| 0 | 1 | 1 × (3/4)³ = 27/64 ≈ 0,422 |
| 1 | 3 | 3 × (1/4) × (3/4)² = 27/64 ≈ 0,422 |
| 2 | 3 | 3 × (1/4)² × (3/4) = 9/64 ≈ 0,141 |
| 3 | 1 | (1/4)³ = 1/64 ≈ 0,016 |

$E(X) = 3 \times 1/4 = 0{,}75$ hijos enfermos de media.

In [ ]:
from scipy import stats
import numpy as np

X = stats.binom(n=3, p=0.25)
for k in range(4):
    print(f"P(X = {k}) = {X.pmf(k):.3f}")
print(f"Media: {X.mean():.2f} · Varianza: {X.var():.4f}")

`scipy.stats` define cada distribución como un objeto con métodos:

| Método | Calcula |
| --- | --- |
| `.pmf(k)` | $P(X = k)$ |
| `.cdf(k)` | $P(X \le k)$ |
| `.sf(k)` | $P(X > k) = 1 - P(X \le k)$ |
| `.mean()`, `.var()` | Esperanza y varianza |
| `.rvs(size, random_state)` | Valores simulados |

### <font color='steelblue'>3.2 Respuesta a un tratamiento</font>

Un antihipertensivo controla la presión arterial en el 60 % de los pacientes. Se trata a 20. ¿Qué probabilidad hay de que **al menos 15** queden controlados?

$$P(X \ge 15) = 1 - P(X \le 14)$$

In [ ]:
import matplotlib.pyplot as plt

X = stats.binom(n=20, p=0.6)
print(f"P(X ≥ 15) = {X.sf(14):.3f}")
print(f"Esperados: {X.mean():.0f} controlados · desviación típica: {X.std():.2f}")

k = np.arange(0, 21)
colores = ["firebrick" if valor >= 15 else "steelblue" for valor in k]
plt.figure(figsize=(7, 4))
plt.bar(k, X.pmf(k), color=colores)
plt.xlabel("Pacientes controlados de 20")
plt.ylabel("Probabilidad")
plt.title("Binomial(n = 20, p = 0,6); en rojo, P(X ≥ 15)")
plt.show()

**Cuidado con «al menos»:** $P(X \ge 15) = 1 - P(X \le 14)$, por eso se usa `.sf(14)` y no `.sf(15)`.

### <font color='steelblue'>3.3 El modelo de IA ante 50 preguntas</font>

Un modelo de lenguaje acierta, según su fabricante, el 80 % de las preguntas de un banco de preguntas de examen. Le planteamos 50 preguntas y acierta 34. Si el 80 % fuera cierto, ¿sería raro acertar 34 o menos?

In [ ]:
aciertos = stats.binom(n=50, p=0.80)
print(f"Aciertos esperados: {aciertos.mean():.0f} de 50")
print(f"P(X ≤ 34) = {aciertos.cdf(34):.4f}")

Si la tasa real fuera del 80 %, acertar 34 o menos ocurriría en solo un 3 % de las series de 50 preguntas. Hay motivos para dudar de esa tasa en **nuestras** preguntas. Este razonamiento es el germen del contraste de hipótesis del Tema 210.

En el Tema 190 veremos que, si la dificultad de las preguntas es muy desigual, la binomial puede no ser el modelo adecuado.

### <font color='steelblue'>3.4 Infecciones al mes: la distribución de Poisson</font>

**A mano**, con λ = 3 infecciones al mes:

- $P(X = 0) = e^{-3} \approx 0{,}050$
- $P(X = 3) = e^{-3} \cdot 3^3 / 3! = 0{,}050 \times 27/6 \approx 0{,}224$

In [ ]:
infecciones = stats.poisson(mu=3)
print(f"P(X = 0) = {infecciones.pmf(0):.3f}")
print(f"P(X = 3) = {infecciones.pmf(3):.3f}")

k = np.arange(0, 13)
plt.figure(figsize=(7, 4))
plt.bar(k, infecciones.pmf(k), color=["firebrick" if valor >= 8 else "steelblue" for valor in k])
plt.xlabel("Infecciones de herida quirúrgica en un mes")
plt.ylabel("Probabilidad")
plt.title("Poisson(λ = 3); en rojo, 8 o más casos")
plt.show()

### <font color='steelblue'>3.5 ¿Es un brote?</font>

In [ ]:
p_8_o_mas = infecciones.sf(7)
print(f"P(X ≥ 8 | λ = 3) = {p_8_o_mas:.4f}")
print(f"Esperaríamos un mes así aproximadamente una vez cada {1 / p_8_o_mas:.0f} meses")

Ver 8 o más infecciones en un mes, si la tasa siguiera siendo 3, ocurriría aproximadamente una vez cada 7 años. Es lo bastante improbable como para **investigar**: revisar técnicas, quirófanos y microorganismos. La estadística no confirma el brote, pero sí indica que el exceso difícilmente es casual.

Comprobamos el resultado simulando 10.000 meses:

In [ ]:
meses = infecciones.rvs(size=10_000, random_state=170)
print(f"Meses con 8 o más infecciones: {(meses >= 8).mean():.4f}")
print(f"Media simulada: {meses.mean():.2f} · Varianza simulada: {meses.var():.2f}")

Observa que la media y la varianza simuladas son casi iguales, como corresponde a una Poisson.

### <font color='steelblue'>3.6 La aproximación de Poisson a la binomial</font>

Un fármaco produce una reacción adversa grave en 1 de cada 1.000 pacientes. En un hospital se trata a 2.000. ¿Qué probabilidad hay de ver al menos un caso?

In [ ]:
exacta = stats.binom(n=2000, p=0.001).sf(0)
aproximada = stats.poisson(mu=2000 * 0.001).sf(0)
print(f"Binomial exacta:       {exacta:.4f}")
print(f"Aproximación de Poisson: {aproximada:.4f}  (λ = np = 2)")
print(f"A mano: 1 − e^(−2) = {1 - np.exp(-2):.4f}")

### <font color='steelblue'>Ampliación: la regla del tres</font>

Si en un estudio con *n* pacientes **no** aparece ningún efecto adverso, ¿podemos decir que el riesgo es cero? No. La **regla del tres** dice que, con un 95 % de confianza, el riesgo real es menor de 3/*n*. Se deduce de la binomial: el riesgo *p* más alto compatible con 0 casos cumple $(1-p)^n = 0{,}05$.

In [ ]:
for n in [30, 100, 1000]:
    p_maximo = 1 - 0.05 ** (1 / n)
    print(f"n = {n:>4}: límite exacto {p_maximo:.4f} · regla del tres {3 / n:.4f}")

Un ensayo con 300 pacientes sin ningún caso de hepatotoxicidad solo permite afirmar que el riesgo es probablemente menor del 1 %. Por eso los efectos adversos raros se descubren después de la comercialización, en la farmacovigilancia.

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Usar la binomial cuando los ensayos no son independientes**, por ejemplo, contar infecciones entre pacientes de la misma habitación durante un brote.

**Usar la binomial cuando *p* no es constante**, como en preguntas de dificultad muy distinta.

**Equivocarse con «al menos» y «más de»:** $P(X \ge 15) = 1 - P(X \le 14)$.

**Aplicar la Poisson a recuentos con varianza mucho mayor que la media** (sobredispersión), habitual cuando la tasa varía entre periodos o unidades.

**Concluir que un riesgo es cero** porque no se ha observado ningún caso.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 170 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** decidir si cada recuento clínico es binomial, de Poisson o ninguno de los dos, y justificarlo.
2. **Misión clínica (con pistas):** calcular a mano y con `scipy.stats` la probabilidad de 4 o más ingresos en la UCI en una noche con λ = 1,8.
3. **Misión reto (con asistente):** simular el número de aciertos de un modelo de IA en 40 preguntas y comparar la distribución simulada con la binomial teórica.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde a mano, sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'Un tratamiento es eficaz en el 70 % de los pacientes. En 10 pacientes, ¿cuántos '
              'esperas que respondan?',
  'type': 'multiple_choice',
  'answers': [{'answer': '7',
               'correct': True,
               'feedback': 'Correcto: la esperanza de una binomial es np = 10 × 0,7.'},
              {'answer': '10',
               'correct': False,
               'feedback': 'No todos responden: la probabilidad de respuesta es 0,7.'},
              {'answer': '2,1', 'correct': False, 'feedback': 'Esa es la varianza, np(1 − p).'},
              {'answer': '0,7',
               'correct': False,
               'feedback': 'Esa es la probabilidad de que responda un paciente.'}]},
 {'question': '¿Qué recuento sigue mejor una distribución de Poisson?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Número de llamadas al 112 por cardiopatía isquémica en una hora',
               'correct': True,
               'feedback': 'Correcto: sucesos independientes en un intervalo de tiempo con una '
                           'tasa media.'},
              {'answer': 'Número de curados de 30 pacientes tratados',
               'correct': False,
               'feedback': 'Tiene un número fijo de ensayos: es binomial.'},
              {'answer': 'Peso de un recién nacido',
               'correct': False,
               'feedback': 'Es una variable continua.'},
              {'answer': 'Grupo sanguíneo de un donante',
               'correct': False,
               'feedback': 'Es una variable categórica.'}]},
 {'question': 'En una Poisson con λ = 2, ¿cuánto vale P(X = 0)?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'e⁻² ≈ 0,135',
               'correct': True,
               'feedback': 'Correcto: e^(−λ) λ⁰ / 0! = e^(−2).'},
              {'answer': '0',
               'correct': False,
               'feedback': 'Que no ocurra ningún suceso es perfectamente posible.'},
              {'answer': '0,5', 'correct': False, 'feedback': 'Revisa la fórmula: e^(−2) ≈ 0,135.'},
              {'answer': '2',
               'correct': False,
               'feedback': 'Una probabilidad no puede ser mayor que 1.'}]},
 {'question': '¿Cómo se calcula P(X ≥ 6) con scipy.stats para una binomial?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'X.sf(5)',
               'correct': True,
               'feedback': 'Correcto: sf(k) = P(X > k), y X > 5 equivale a X ≥ 6.'},
              {'answer': 'X.sf(6)', 'correct': False, 'feedback': 'sf(6) = P(X > 6) = P(X ≥ 7).'},
              {'answer': 'X.cdf(6)', 'correct': False, 'feedback': 'cdf(6) = P(X ≤ 6).'},
              {'answer': 'X.pmf(6)',
               'correct': False,
               'feedback': 'pmf(6) = P(X = 6), solo ese valor.'}]},
 {'question': 'En un ensayo con 150 pacientes no aparece ningún caso de anafilaxia. ¿Qué puede '
              'afirmarse con un 95 % de confianza?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Que el riesgo es probablemente menor del 2 %',
               'correct': True,
               'feedback': 'Correcto: regla del tres, 3 / 150 = 0,02.'},
              {'answer': 'Que el riesgo es cero',
               'correct': False,
               'feedback': 'No haber visto casos no demuestra que no existan.'},
              {'answer': 'Que el riesgo es menor de 1 de cada 150',
               'correct': False,
               'feedback': 'La regla del tres da 3/n, no 1/n.'},
              {'answer': 'Nada: con cero casos no se puede calcular nada',
               'correct': False,
               'feedback': 'La regla del tres da un límite superior.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Una variable aleatoria discreta se describe con su función de probabilidad, su esperanza y su varianza.
- **Binomial**: éxitos en *n* ensayos independientes con la misma *p*; media *np*.
- **Poisson**: sucesos en un intervalo con tasa λ; media y varianza iguales a λ.
- Un recuento muy improbable bajo el modelo habitual es una señal para investigar, no una prueba definitiva.
- No observar casos no demuestra que el riesgo sea cero (regla del tres).

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Variable aleatoria discreta | Variable que toma valores aislados, normalmente recuentos |
| Función de probabilidad | Probabilidad de cada valor de la variable |
| Función de distribución | Probabilidad acumulada hasta un valor |
| Esperanza | Valor medio de una variable aleatoria a largo plazo |
| Distribución binomial | Número de éxitos en *n* ensayos independientes con probabilidad constante |
| Distribución de Poisson | Número de sucesos en un intervalo con una tasa media constante |
| Tasa (λ) | Número medio de sucesos por unidad de tiempo o espacio |
| Sobredispersión | Situación en la que la varianza observada es mayor que la prevista por el modelo |
| Regla del tres | Si no hay casos en *n* sujetos, el riesgo es probablemente menor de 3/*n* |

## <font color='steelblue'>Para saber más</font>

- Hanley JA, Lippman-Hand A. If nothing goes wrong, is everything all right? Interpreting zero numerators. *JAMA*. 1983;249(13):1743-1745.
- [Documentación de scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

## <font color='steelblue'>Retos de este tema en la app</font>

Practica lo que has visto con las tres misiones del Tema 170 en la app **Bioestadística**. Cuentan para la evaluación continua.

1. **Básica (sin ayuda):** ¿Binomial, Poisson o ninguna?
2. **Clínica (con pistas):** Probabilidad de 4 o más ingresos en la UCI en una noche
3. **Reto (con asistente de IA):** Simular los aciertos de un modelo de IA en 40 preguntas

<a href="https://fborrasumh.github.io/bioestadistica/#tema-170" target="_blank" style="display:inline-block;padding:10px 18px;background:#C8461F;color:#fff;border-radius:8px;font-weight:bold;text-decoration:none">Abrir los retos del Tema 170 en la app ↗</a>

---
*Tema 170 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*